# 16 Groundwater head beside the Tule Pond section

```
Author: Danielle Lindsay
Environment: opera_disp_feb13. Run time: seconds.
```

**What this notebook does.** The seasonal change in groundwater head in the aquifers beside the Tule Pond
section, from the Alameda County Water District's (ACWD) public monitoring data, and its size as a pore-pressure
change on the fault, for comparison with the stress drop of the slow slip events (SSEs).

1. **The Spring and Fall 2025 well tables** of the ACWD 2025 Groundwater Monitoring Report (Chen 2026, Appendices
   E and F; `data_prep/acwd_gwmr_tables.py`) are joined by State Well Number. Spring minus fall is the seasonal
   high minus the seasonal low of water year 2024/25. The wells beside the section are those in the township
   sections the section crosses (T4S R1W 21, 22, 27, 28, 29, 33, 34); their side of the fault is the aquifer code
   (ACWD's AHF aquifer lies east of the Hayward Fault, every other aquifer west of it).
2. **The two ACWD indicator wells nearest the section** (27D008 east, 29A006 west, ~0.8 km from the trace, weekly
   levels since 1990/1991) give the range of head in every water year since 1991, and the spring-to-fall change,
   with the water years of the InSAR SSEs flagged.
3. A head change of $\Delta h$ metres of water is a pore-pressure change $\Delta p = \rho_w g \Delta h$,
   9.81 kPa per metre, at the depth of the well screen (hydrostatic, no storage change in the aquitards).

The numbers go to the registry as `16_*` and the stats sidecar `16_aquifer_head_stats.txt`.

## 1. Imports and settings

In [1]:
import os
import re

import insar_utils as utils
from TulePond_filepaths import common_paths, paths_out, repo_dir

import numpy as np
import pandas as pd

notebook = "16_aquifer_head"
out_dir = os.path.join(common_paths["out_dir"], notebook)
os.makedirs(out_dir, exist_ok=True)
REG = utils.Registry(notebook, common_paths, repo_dir)

gwmr_dir = common_paths["acwd_gwmr_dir"]
well_dir = common_paths["acwd_well_dir"]
spring_file = os.path.join(gwmr_dir, "spring_2025.csv")
fall_file = os.path.join(gwmr_dir, "fall_2025.csv")
locations_file = os.path.join(well_dir, "well_locations.csv")
well_files = {"4S/1W-27D008": os.path.join(well_dir, "WL 4S1W27D008.csv"),      # east (AHF aquifer)
              "4S/1W-29A006": os.path.join(well_dir, "WL 4S1W29A006.csv")}      # west (Newark aquifer)
events_file = os.path.join(paths_out["04_event_detection"], "creep_events.csv")   # the InSAR SSE onsets (04)

FT_M = 0.3048
KPA_PER_M = 9.81                   # 1 m of water: 1000 kg/m3 x 9.81 m/s2
near_sections = ("21", "22", "27", "28", "29", "33", "34")   # T4S R1W sections along the SSE section (km 56-62)
east_aquifers = {"AHF"}            # ACWD: the AHF sub-basin is east of the Hayward Fault; N, C, F, D (and combinations) are west
spring_months, fall_months = (3, 4), (9, 10)                 # ACWD's spring and fall measurement programmes
min_obs_per_wy = 40                # weekly series: water years with fewer measurements are left out
table_columns = ["well", "alt_id", "aquifer", "owner", "date", "reference_elevation_ft", "water_elevation_ft"]

REG.inputs(spring_file, fall_file, locations_file, events_file, *well_files.values())
print("inputs:", gwmr_dir, well_dir, events_file, sep="\n  ")

Registry for 16_aquifer_head: run ea66373d, git 37f49df+
inputs:
  /Users/dlindsay/work/proj_TulePond_DISP/external_inputs/ACWD_GWMR_2025
  /Users/dlindsay/work/proj_TulePond_DISP/external_inputs/ACWD_water_well_data
  /Users/dlindsay/work/proj_TulePond_DISP/outputs/04_event_detection/creep_events.csv


## 2. The Spring and Fall 2025 tables, joined by well

Rows without a numeric water level (not measured, or artesian) are dropped. The join keeps the wells measured in
both programmes; the aquifer code must agree between the two tables.

In [2]:
def read_table(path, months):
    """One ACWD seasonal table (csv from data_prep/acwd_gwmr_tables.py) as a frame indexed by well, levels in ft."""
    table = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(table.columns) == table_columns, f"{path}: unexpected columns {list(table.columns)}"
    assert table.well.is_unique, f"{path}: a well appears twice"
    measured = table[table.water_elevation_ft.str.match(r"^-?\d+(\.\d+)?$")].copy()
    measured["level_ft"] = measured.water_elevation_ft.astype(float)
    measured["date"] = pd.to_datetime(measured.date, format="%m/%d/%Y")
    assert measured.date.dt.month.isin(months).all(), f"{path}: a measurement outside months {months}"
    print(f"{os.path.basename(path)}: {len(table)} wells, {len(measured)} with a level, dates {measured.date.min().date()} to {measured.date.max().date()}")
    return measured.set_index("well")


spring = read_table(spring_file, spring_months)
fall = read_table(fall_file, fall_months)
both = spring.join(fall, how="inner", lsuffix="_spring", rsuffix="_fall")
assert (both.aquifer_spring == both.aquifer_fall).all(), "aquifer code differs between the spring and fall tables"
both = both.rename(columns={"aquifer_spring": "aquifer"}).drop(columns=["aquifer_fall"])
both["dh_ft"] = both.level_ft_spring - both.level_ft_fall           # seasonal high minus seasonal low
both["dh_m"] = both.dh_ft * FT_M
both["dp_kpa"] = both.dh_m * KPA_PER_M
both["township"] = both.index.str[:5]
both["section"] = both.index.str[6:8]
both["side"] = np.where(both.aquifer.isin(east_aquifers), "east", "west")
both["near_section"] = (both.township == "4S/1W") & both.section.isin(near_sections)
print(f"{len(both)} wells measured in both programmes; {both.near_section.sum()} beside the section "
      f"({(both.near_section & (both.side == 'east')).sum()} east, {(both.near_section & (both.side == 'west')).sum()} west)")
# the aquifer code must separate the two sides: no township section beside the fault holds both AHF and BHF wells
near = both[both.near_section]
shared = set(near[near.side == "east"].section) & set(near[near.side == "west"].section)
assert not shared, f"sections with wells on both sides by aquifer code: {shared}"
both.to_csv(os.path.join(out_dir, "spring_fall_2025.csv"))
print("Saved", os.path.join(out_dir, "spring_fall_2025.csv"))

spring_2025.csv: 295 wells, 245 with a level, dates 2025-03-24 to 2025-04-03
fall_2025.csv: 295 wells, 246 with a level, dates 2025-09-15 to 2025-09-26
236 wells measured in both programmes; 55 beside the section (23 east, 32 west)
Saved /Users/dlindsay/work/proj_TulePond_DISP/outputs/16_aquifer_head/spring_fall_2025.csv


## 3. Spring-to-fall head change beside the section

By side of the fault, and on the west side by aquifer (Newark N is the shallowest, D the deepest, >= ~120 m).

In [3]:
def summary(frame, by):
    """n, median, min and max of the head change (m) and pore-pressure change (kPa), per group of `by`."""
    out = frame.groupby(by).agg(n=("dh_m", "size"), dh_m_median=("dh_m", "median"), dh_m_min=("dh_m", "min"), dh_m_max=("dh_m", "max"),
                                dp_kpa_median=("dp_kpa", "median"), dp_kpa_min=("dp_kpa", "min"), dp_kpa_max=("dp_kpa", "max"))
    assert (out.n > 0).all()
    return out


by_side = summary(near, "side")
by_aquifer_west = summary(near[near.side == "west"], "aquifer")
basin = summary(both, "side")
print("beside the section (T4S R1W sections", ", ".join(near_sections) + "):")
print(by_side.round(2).to_string())
print("\nwest side by aquifer:")
print(by_aquifer_west.round(2).to_string())
print("\nwhole basin, by side:")
print(basin.round(2).to_string())
indicator = both.loc[list(well_files)]
print("\nindicator wells:")
print(indicator[["alt_id_spring", "aquifer", "side", "section", "level_ft_spring", "level_ft_fall", "dh_m", "dp_kpa"]].round(2).to_string())
assert (indicator.side.values == ["east", "west"]).all(), "the indicator wells' sides by aquifer code do not match their positions"
by_side.to_csv(os.path.join(out_dir, "spring_fall_2025_by_side.csv"))

beside the section (T4S R1W sections 21, 22, 27, 28, 29, 33, 34):
       n  dh_m_median  dh_m_min  dh_m_max  dp_kpa_median  dp_kpa_min  dp_kpa_max
side                                                                            
east  23         2.84      0.34      3.36          27.84        3.32       32.98
west  32         1.88      1.04      1.93          18.40       10.23       18.93

west side by aquifer:
         n  dh_m_median  dh_m_min  dh_m_max  dp_kpa_median  dp_kpa_min  dp_kpa_max
aquifer                                                                           
C        6         1.89      1.85      1.93          18.52       18.12       18.93
CF       4         1.91      1.90      1.92          18.72       18.63       18.84
D        6         1.89      1.48      1.90          18.52       14.50       18.66
F        8         1.88      1.82      1.90          18.40       17.82       18.69
N        8         1.42      1.04      1.84          13.92       10.23       18.03

whole

## 4. The two indicator wells, every water year since 1991

ACWD's level series (weekly to monthly). For each water year (1 October to 30 September): the number of
measurements, the highest and lowest level, their difference (the annual range), and the spring (March-April)
minus fall (September-October) means, the same quantity as section 3. The water years of the five InSAR SSEs are
flagged from the onsets of notebook 04.

In [4]:
def water_year(dates):
    dates = pd.DatetimeIndex(dates)
    return dates.year + (dates.month >= 10)


def read_well(path, well):
    series = pd.read_csv(path, encoding="utf-8-sig", dtype=str)
    assert list(series.columns) == ["MeasurementDate", "ReferencePointElevation", "H2OElevation", "WellName"], f"{path}: {list(series.columns)}"
    assert (series.WellName == well).all(), f"{path}: a row of another well"
    series["date"] = pd.to_datetime(series.MeasurementDate, format="%d/%m/%Y")
    series["level_ft"] = series.H2OElevation.astype(float)
    assert series.date.is_monotonic_increasing, f"{path}: dates not in order"
    series["wy"] = water_year(series.date)
    print(f"{well}: {len(series)} measurements, {series.date.min().date()} to {series.date.max().date()}, "
          f"median spacing {series.date.diff().median().days} d")
    return series


events = pd.read_csv(events_file)
insar_onsets = pd.to_datetime(events[events.segment == "TULEPOND"].onset_lo_utc)
sse_water_years = sorted(set(water_year(insar_onsets)))
assert len(sse_water_years) == len(insar_onsets), "two InSAR SSEs in one water year"
print("InSAR SSE water years:", sse_water_years)

locations = pd.read_csv(locations_file, comment="#")
by_wy = []
for well, path in well_files.items():
    series = read_well(path, well)
    side = locations.loc[locations.well == well, "side"].item()
    for wy, year in series.groupby("wy"):
        spring_levels = year[year.date.dt.month.isin(spring_months)].level_ft
        fall_levels = year[year.date.dt.month.isin(fall_months)].level_ft
        if len(year) < min_obs_per_wy or spring_levels.empty or fall_levels.empty:
            continue
        by_wy.append({"well": well, "side": side, "wy": wy, "n": len(year),
                      "max_ft": year.level_ft.max(), "min_ft": year.level_ft.min(),
                      "range_m": (year.level_ft.max() - year.level_ft.min()) * FT_M,
                      "spring_minus_fall_m": (spring_levels.mean() - fall_levels.mean()) * FT_M,
                      "sse_year": wy in sse_water_years})
by_wy = pd.DataFrame(by_wy)
by_wy["range_kpa"] = by_wy.range_m * KPA_PER_M
by_wy["spring_minus_fall_kpa"] = by_wy.spring_minus_fall_m * KPA_PER_M
assert not by_wy.empty
by_wy.to_csv(os.path.join(out_dir, "indicator_wells_by_water_year.csv"), index=False)
insar_era = by_wy[by_wy.wy >= min(sse_water_years)]
for well, frame in by_wy.groupby("well"):
    era = frame[frame.wy >= min(sse_water_years)]
    print(f"\n{well} ({frame.side.iloc[0]}): {len(frame)} complete water years {frame.wy.min()}-{frame.wy.max()}")
    print(f"  annual range: median {frame.range_m.median():.2f} m (all years), {era.range_m.median():.2f} m (WY{era.wy.min()}-{era.wy.max()}), "
          f"{era[era.sse_year].range_m.median():.2f} m (SSE years), {era[~era.sse_year].range_m.median():.2f} m (other years)")
    print(f"  spring minus fall: median {era.spring_minus_fall_m.median():.2f} m (WY{era.wy.min()}-{era.wy.max()}), "
          f"min {era.spring_minus_fall_m.min():.2f}, max {era.spring_minus_fall_m.max():.2f}")
    print(era[["wy", "n", "range_m", "spring_minus_fall_m", "range_kpa", "sse_year"]].round(2).to_string(index=False))

InSAR SSE water years: [2017, 2019, 2021, 2023, 2025]
4S/1W-27D008: 1924 measurements, 1990-10-02 to 2026-09-01, median spacing 7 d
4S/1W-29A006: 2192 measurements, 1991-10-09 to 2026-09-01, median spacing 7 d

4S/1W-27D008 (east): 36 complete water years 1991-2026
  annual range: median 4.27 m (all years), 3.53 m (WY2017-2026), 3.79 m (SSE years), 2.52 m (other years)
  spring minus fall: median 2.30 m (WY2017-2026), min 0.66, max 4.00
  wy  n  range_m  spring_minus_fall_m  range_kpa  sse_year
2017 54     3.79                 2.71      37.17      True
2018 53     2.52                 1.10      24.76     False
2019 55     4.72                 3.01      46.35      True
2020 56     2.30                 0.66      22.52     False
2021 54     3.77                 2.50      36.99      True
2022 55     4.37                 4.00      42.91     False
2023 54     5.92                 2.93      58.04      True
2024 54     2.15                 1.60      21.08     False
2025 53     3.14            

## 5. Registry: the stats sidecar `16_aquifer_head_stats.txt` and the `16_*` numbers

In [5]:
lines = [f"ACWD 2025 GWMR Spring/Fall 2025 tables: {len(both)} wells with both levels; beside the section (T4S R1W {', '.join(near_sections)}): "
         f"{len(near)} wells, {(near.side == 'east').sum()} east (AHF) and {(near.side == 'west').sum()} west",
         "spring minus fall 2025 by side (m of head; kPa at 9.81 kPa/m):", by_side.round(3).to_string(),
         "west side by aquifer:", by_aquifer_west.round(3).to_string(),
         "indicator wells 2025:", indicator[["aquifer", "side", "level_ft_spring", "level_ft_fall", "dh_m", "dp_kpa"]].round(3).to_string(),
         f"indicator-well series: water years with >= {min_obs_per_wy} measurements and both seasons; InSAR SSE water years {sse_water_years}",
         by_wy.round(3).to_string(index=False)]
sidecar = REG.sidecar("16_aquifer_head", lines)
REG.register("16_n_wells_both_seasons", len(both), "count", 0, where="M12", cell="tables", sidecar=sidecar)
for side in ("east", "west"):
    s = by_side.loc[side]
    REG.register(f"16_near_{side}_n", int(s.n), "count", 0, where="M12", cell="tables", sidecar=sidecar)
    REG.register(f"16_near_{side}_dh_median_m", s.dh_m_median, "m", 2, where="P3", cell="tables", sidecar=sidecar)
    REG.register(f"16_near_{side}_dh_min_m", s.dh_m_min, "m", 2, where="P3", cell="tables", sidecar=sidecar)
    REG.register(f"16_near_{side}_dh_max_m", s.dh_m_max, "m", 2, where="P3", cell="tables", sidecar=sidecar)
    REG.register(f"16_near_{side}_dp_median_kpa", s.dp_kpa_median, "kPa", 0, where="P3", cell="tables", sidecar=sidecar)
    REG.register(f"16_near_{side}_dp_min_kpa", s.dp_kpa_min, "kPa", 0, where="P3", cell="tables", sidecar=sidecar)
    REG.register(f"16_near_{side}_dp_max_kpa", s.dp_kpa_max, "kPa", 0, where="P3", cell="tables", sidecar=sidecar)
if "D" in by_aquifer_west.index:
    REG.register("16_near_deep_aquifer_dh_median_m", by_aquifer_west.loc["D"].dh_m_median, "m", 2, where="M12", cell="tables", sidecar=sidecar)
for well, label in (("4S/1W-27D008", "east"), ("4S/1W-29A006", "west")):
    REG.register(f"16_indicator_{label}_dh_2025_m", both.loc[well, "dh_m"], "m", 2, where="M12", cell="tables", sidecar=sidecar)
    REG.register(f"16_indicator_{label}_dp_2025_kpa", both.loc[well, "dp_kpa"], "kPa", 0, where="M12", cell="tables", sidecar=sidecar)
    era = by_wy[(by_wy.well == well) & (by_wy.wy >= min(sse_water_years))]
    REG.register(f"16_indicator_{label}_wy_first", int(era.wy.min()), "water year", 0, where="M12", cell="series", sidecar=sidecar)
    REG.register(f"16_indicator_{label}_wy_last", int(era.wy.max()), "water year", 0, where="M12", cell="series", sidecar=sidecar)
    REG.register(f"16_indicator_{label}_range_median_m", era.range_m.median(), "m", 2, where="M12", cell="series", sidecar=sidecar)
    REG.register(f"16_indicator_{label}_range_min_m", era.range_m.min(), "m", 2, where="M12", cell="series", sidecar=sidecar)
    REG.register(f"16_indicator_{label}_range_max_m", era.range_m.max(), "m", 2, where="M12", cell="series", sidecar=sidecar)
    REG.register(f"16_indicator_{label}_range_sse_years_median_m", era[era.sse_year].range_m.median(), "m", 2, where="M12", cell="series", sidecar=sidecar)
    REG.register(f"16_indicator_{label}_sf_median_m", era.spring_minus_fall_m.median(), "m", 2, where="M12", cell="series", sidecar=sidecar)
REG.register("16_kpa_per_m", KPA_PER_M, "kPa/m", 2, where="M12", cell="settings", sidecar=sidecar)
REG.render()

Saved stats to: /Users/dlindsay/work/proj_TulePond_DISP/outputs/16_aquifer_head/16_aquifer_head_stats.txt
  registered 16_n_wells_both_seasons = 236 count  -> M12


  registered 16_near_east_n = 23 count  -> M12


  registered 16_near_east_dh_median_m = 2.84 m  -> P3
  registered 16_near_east_dh_min_m = 0.34 m  -> P3


  registered 16_near_east_dh_max_m = 3.36 m  -> P3


  registered 16_near_east_dp_median_kpa = 28 kPa  -> P3
  registered 16_near_east_dp_min_kpa = 3 kPa  -> P3


  registered 16_near_east_dp_max_kpa = 33 kPa  -> P3


... (16 outputs left out)

  registered 16_indicator_west_range_sse_years_median_m = 2.11 m  -> M12
  registered 16_indicator_west_sf_median_m = 1.48 m  -> M12


  registered 16_kpa_per_m = 9.81 kPa/m  -> M12


  01_prep_inputs: its latest run registered no numbers; its numbers are left out
Wrote the numbers to: /Users/dlindsay/work/proj_TulePond_DISP/outputs/paper_numbers.md


'/Users/dlindsay/work/proj_TulePond_DISP/outputs/paper_numbers.md'